# **Norwegian News Articles**
Project for TDT4310

By: Malin Haugland Høli

## News Framing

### *Imports*

In [1]:
import pandas as pd
import spacy
from pathlib import Path

from transformers import pipeline
from tqdm import tqdm
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

c:\NTNU\V26\TDT4310\Project\exploring-norwegian-news-sources\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### *Import dataset*

In [2]:
df = pd.read_parquet("../combined-2019.parquet")
df.head(3)

,file,url,source,date,author,gender,class1,class2,language,title,ingress,text,word_count,sentence_count
0,VG-20190101-16-personer-fikk-oye.xml,http://www.vg.no/nyheter/innenriks/i/jPyoyb/16...,VG,2019-01-01 09:12,Halvor Bjørntvedt,Male,"nyheter,innenriks",None,Bokmål,16 personer fikk øyeskader av fyrverkeri,Fem fikk alvorlige skader etter gårsdagens nyt...,"Tallene, som viser øyeskader på landsbasis, vi...",209,13
1,VG-20190101-47-plass-for-sundby-.xml,http://www.vg.no/sport/langrenn/i/3j8ra0/47-pl...,VG,2019-01-01 10:19,"Nils Mangelrød, Jostein Overvik",Male,"sport,langrenn",None,Bokmål,47. plass for Sundby - nekter å prate etter ne...,VAL MÜSTAIR (VG) Martin Johnsrud nektet å prat...,Den tidligere Tour de Ski-vinneren var svært l...,533,42
2,VG-20190101-94-drap-pa-journalis.xml,http://www.vg.no/nyheter/utenriks/i/J19kWP/94-...,VG,2019-01-01 21:37,Mikal Hem,Male,"nyheter,utenriks","Drap, Ytringsfrihet, Journalistikk, Korrupsjon...",Bokmål,94 drap på journalister i 2018: Skal ha blitt ...,Antall drepte journalister i Vesten har skutt ...,25. februar ble Jan Kuciak og hans forlovede M...,539,44


---
### *Sample 1000 articles from each source*

In [3]:
samples = []
for source in df["source"].unique():
    sample = df[df["source"] == source].sample(1000, random_state=42)
    print(f"Sampled {len(sample)} articles from {source}")
    samples.append(sample)

sample_df = pd.concat(samples)  # original indexes preserved
assert len(set(sample_df.index)) == len(sample_df)  # check for duplicates

Sampled 1000 articles from VG
Sampled 1000 articles from AA
Sampled 1000 articles from AP
Sampled 1000 articles from BT
Sampled 1000 articles from DA
Sampled 1000 articles from DN
Sampled 1000 articles from DB
Sampled 1000 articles from FV
Sampled 1000 articles from NL
Sampled 1000 articles from SA


---
### *POS and Dependency Tags*

In [4]:
nlp = spacy.load("nb_core_news_lg", disable=["ner"])

In [5]:
def analyze_article(text):
    """
    Analyze the article text to extract linguistic features.
    Returns a dictionary with verb ratio, subject ratio, passive voice ratio, and passive of verbs ratio.
    """
    doc = nlp(text)

    verbs = []
    subjects = []
    passive_count = 0

    for token in doc:
        if token.pos_ == "VERB":
            verbs.append(token.lemma_.lower())

        if token.dep_ in ("nsubj", "nsubj:pass"):
            subjects.append(token.lemma_.lower())

        if token.dep_ == "aux:pass":
            passive_count += 1

    token_count = len(doc)

    return {
        "verb_ratio": len(verbs) / token_count if token_count > 0 else 0,
        "subject_ratio": len(subjects) / token_count if token_count > 0 else 0,
        "passive_ratio": passive_count / token_count if token_count > 0 else 0,
        "passive_of_verbs": passive_count / len(verbs) if len(verbs) > 0 else 0,
    }

In [6]:
analysis_results = []

for idx, row in tqdm(sample_df.iterrows(), total=sample_df.shape[0]):
    analysis_results.append(analyze_article(row["text"]))

analysis_df = pd.DataFrame(analysis_results)
analysis_df.index = sample_df.index  # align indexes for merging
analysis_df = pd.merge(sample_df, analysis_df, left_index=True, right_index=True)
analysis_df.head(3)

100%|██████████| 10000/10000 [11:46<00:00, 14.16it/s]


,file,url,source,date,author,gender,class1,class2,language,title,ingress,text,word_count,sentence_count,verb_ratio,subject_ratio,passive_ratio,passive_of_verbs
5643,VG-20190423-ekstra-bladet-danske.xml,http://www.vg.no/nyheter/utenriks/i/awOaE7/eks...,VG,2019-04-23 12:57,"Ingvill Dybfest Dahl, Ingeborg Huse Amundsen, ...",Female,"nyheter,utenriks",None,Bokmål,Ekstra Bladet: Danske Anders Holch Povlsen på ...,Den danske klesmilliardæren som mistet tre av ...,"Det skriver danske Ekstra Bladet, som siterer ...",439,27,0.099804,0.088063,0.007828,0.078431
6808,VG-20190516-sarah-jessica-parker.xml,http://www.vg.no/rampelys/film/i/rA5exK/sarah-...,VG,2019-05-16 07:37,Catherine Gonsholt Ighanian,Female,"rampelys,film",None,Bokmål,Sarah Jessica Parker raser mot ukeblad – viser...,Ukebladet som sier at de jobber med en sak om ...,Den tidligere «Sex og singelliv»-stjernen har ...,435,26,0.095785,0.086207,0.001916,0.020000
4593,VG-20190401-rotteracet-om-champi.xml,http://www.vg.no/sport/fotball/i/opEPLm/rotter...,VG,2019-04-01 10:20,Ole Kristian Strøm,Male,"sport,fotball",None,Bokmål,Rotteracet om Champions League-plass: – Det er...,Mandag kveld har Arsenal sjansen til å presse ...,Mandagens oddstips finner du her\n– Jeg våger ...,611,45,0.098571,0.092857,0.004286,0.043478


#### *Linguistic features by source*

In [7]:
source_stats = analysis_df.groupby("source").agg({
    "passive_ratio": "mean",
    "subject_ratio": "mean",
    "verb_ratio": "mean",
    "passive_of_verbs": "mean"
})
print(source_stats)

        passive_ratio  subject_ratio  verb_ratio  passive_of_verbs
source                                                            
AA           0.005571       0.094068    0.102430          0.053794
AP           0.005272       0.093220    0.102769          0.051138
BT           0.004702       0.093543    0.103314          0.045172
DA           0.005144       0.092860    0.103919          0.049007
DB           0.004786       0.094752    0.104951          0.045786
DN           0.003562       0.090372    0.100508          0.035075
FV           0.004851       0.092375    0.102397          0.047441
NL           0.005889       0.091954    0.101774          0.056576
SA           0.004196       0.091370    0.101193          0.040898
VG           0.005370       0.097687    0.107851          0.050151


---
### *Sentiment analysis*

In [8]:
pipe = pipeline(
    "text-classification",
    model="malinhauglandh/norec-sentiment-norbert3-small",
    trust_remote_code=True
)

Loading weights: 100%|██████████| 128/128 [00:00<00:00, 1677.78it/s]


In [10]:
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

texts = sample_df["text"].tolist()

outputs = []

for chunk in tqdm(chunk_list(texts, 20), total=(len(texts) + 19) // 20):  # chunk size = 20
    preds = pipe(
        chunk,
        batch_size=4,
        truncation=True,
        max_length=256
    )
    outputs.extend(preds)

100%|██████████| 500/500 [39:38<00:00,  4.76s/it]


In [11]:
analysis_df["label"] = [o["label"] for o in outputs]
analysis_df["score"] = [o["score"] for o in outputs]
analysis_df.head(3)

,file,url,source,date,author,gender,class1,class2,language,title,ingress,text,word_count,sentence_count,verb_ratio,subject_ratio,passive_ratio,passive_of_verbs,label,score
5643,VG-20190423-ekstra-bladet-danske.xml,http://www.vg.no/nyheter/utenriks/i/awOaE7/eks...,VG,2019-04-23 12:57,"Ingvill Dybfest Dahl, Ingeborg Huse Amundsen, ...",Female,"nyheter,utenriks",None,Bokmål,Ekstra Bladet: Danske Anders Holch Povlsen på ...,Den danske klesmilliardæren som mistet tre av ...,"Det skriver danske Ekstra Bladet, som siterer ...",439,27,0.099804,0.088063,0.007828,0.078431,neutral,0.540266
6808,VG-20190516-sarah-jessica-parker.xml,http://www.vg.no/rampelys/film/i/rA5exK/sarah-...,VG,2019-05-16 07:37,Catherine Gonsholt Ighanian,Female,"rampelys,film",None,Bokmål,Sarah Jessica Parker raser mot ukeblad – viser...,Ukebladet som sier at de jobber med en sak om ...,Den tidligere «Sex og singelliv»-stjernen har ...,435,26,0.095785,0.086207,0.001916,0.020000,neutral,0.640248
4593,VG-20190401-rotteracet-om-champi.xml,http://www.vg.no/sport/fotball/i/opEPLm/rotter...,VG,2019-04-01 10:20,Ole Kristian Strøm,Male,"sport,fotball",None,Bokmål,Rotteracet om Champions League-plass: – Det er...,Mandag kveld har Arsenal sjansen til å presse ...,Mandagens oddstips finner du her\n– Jeg våger ...,611,45,0.098571,0.092857,0.004286,0.043478,neutral,0.488746


In [12]:
analysis_df["label"].value_counts()

label
neutral     7400
positive    2368
negative     232
Name: count, dtype: int64

### *Print labels*

In [13]:
# print 3 examples of each label
for label in analysis_df["label"].unique():
    print(f"\n--- {label.upper()} EXAMPLES ---")
    examples = analysis_df[analysis_df["label"] == label].head(3)
    for idx, row in examples.iterrows():
        print(f"\nExample {idx} (score: {row['score']:.4f}):")
        print(row["text"][:2000])


--- NEUTRAL EXAMPLES ---

Example 5643 (score: 0.5403):
Det skriver danske Ekstra Bladet, som siterer kilder ved akuttmottaket.
– Et av familiemedlemmene var skadet, men er nå utskrevet fra sykehus, bekrefter Danmarks ambassadør i India, Peter Taksøe-Jensen, overfor VG.
På National Hospital of Sri Lanka bekrefter flere kilder at en fra Holch Povlsen-familien var såret i angrepet på luksushotellet Shangri-La og innlagt. En medarbeider ved akuttmottaket opplyser at det var snakk om en voksen dansk mann.
Bakgrunn: Dette vet vi om angrepene
Den 46 år gamle Bestseller-milliardæren og kona Anne Holch Povlsen (40) var på påskeferie sammen med barna på Sri Lanka, da landet i Det indiske hav ble rammet av åtte eksplosjoner søndag.
Angrepene har så langt kostet 321 mennesker livet, og antall døde og skadede fortsetter å stige. Tre av barna til det danske ekteparet var blant de drepte.
Torsdag skal det etter planen holdes et fakkeltog i Stavdrup i Danmark, hvor Bestseller-familien bor. Dette bli

In [14]:
# print 10 examples of each label with highest confidence
for label in analysis_df["label"].unique():
    print(f"\n--- {label.upper()} HIGH CONFIDENCE EXAMPLES ---")
    examples = analysis_df[analysis_df["label"] == label].sort_values("score", ascending=False).head(10)
    for idx, row in examples.iterrows():
        print(f"\nExample {idx} (score: {row['score']:.4f}):")
        print(row["text"][:2000])


--- NEUTRAL HIGH CONFIDENCE EXAMPLES ---

Example 55565 (score: 0.9929):
4
TV-drama
«Skitten snø»
NRK.no, premiere 31. oktober
En ungdomsserie som tar opp voldtekt, moralpoliti og skamkulturen som preger deler av et innvandrermiljø er ikke dagligdagse saker på NRK. «Skitten snø» byr på tematikken, i formm av dramaserien basert på boka til Mahmona Khan som kom for åtte år siden. «Skitten snø» var en kritikerrost ungdomsroman, omtalt som «en solid skjønnlitteratur debut» i denne avisen. TV-versjonen skal først og fremst være en spenningshistorie for unge seere som berører overgrep, vennskap, det å være offer for et overgrep, men enda viktigere det å ta tilbake kontroll over eget liv. Men selv om serien kommer med plakaten «for 15 år pluss» har «Skitten snø» også ambisjon om å nå målgruppas foreldre, å skape refleksjon og gi i innblikk i et samfunnsproblem som trenger oppmerksomhet.
Det er unge innvandrerjenter som har hovedrollen, men «Skitten snø» byr også en nedpå, flersidig skildring

In [ ]:
# check which source has the most positive/negative articles
sentiment_by_source = analysis_df.groupby("source")["label"].value_counts(normalize=True).unstack().fillna(0)
print("\nSentiment distribution by source:")
print(sentiment_by_source)


Sentiment distribution by source:
label   negative  neutral  positive
source                             
AA         0.006    0.787     0.207
AP         0.014    0.785     0.201
BT         0.029    0.669     0.302
DA         0.025    0.772     0.203
DB         0.038    0.733     0.229
DN         0.018    0.767     0.215
FV         0.003    0.691     0.306
NL         0.061    0.721     0.218
SA         0.022    0.712     0.266
VG         0.016    0.763     0.221


### *Fædrelandsvennen has the highest distribution of articles labeled as positive*
What are the top 3 positive articles from FV?

In [15]:
# print top 3 positive articles from FV
fv_positive = analysis_df[(analysis_df["source"] == "FV") & (analysis_df["label"] == "positive")].sort_values("score", ascending=False).head(3)
print("\n--- TOP 3 POSITIVE ARTICLES FROM FV ---")
for idx, row in fv_positive.iterrows():
    print(f"\nExample {idx} (score: {row['score']:.4f}):")
    print(row["text"][:2000])


--- TOP 3 POSITIVE ARTICLES FROM FV ---

Example 90578 (score: 0.9903):
Hei dere! Da er vi tilbake fra en sommer vi aldri kommer til å glemme, med oppsetningen ”Fruen fra havet” i Fjæreheia. Vi brukte noen uker både på å lande, og samle tankene etter så mange inntrykk. Det har vært en intens, lærerik, tøff, nydelig og utrolig verdifull prosess.
Vi visste lite om hva vi skulle forvente oss av det vi gikk inn i, bare at det krevde en god del vågemot og troen på at det er sunt å hoppe utfor hoppkanten av og til. Vi angrer ikke ett sekund. Vi er heller litt tomme for ord av takknemlighet og beundring over tilliten vi har fått. Vi tar hatten av for Birgit Amalie Nilssen. For ei rå dame du er! Vi følger nøye med, og tar inn så mye vi kan. Du er så varm og var. Det merkes av hele teamet, tror vi.
Privat
Det har vært mange dyktige mennesker å følge med på under denne reisen. Amalie Krogh i sitt ess, og Tove Kragset som en musikalsk påle – og ikke minst med evnen til å improvisere nye pianopar

### *Nordlys has the highest distribution of articles labeled as negative*
What are the top 3 negative articles from NL?

In [16]:
# print top 3 negative articles from NL
nl_negative = analysis_df[(analysis_df["source"] == "NL") & (analysis_df["label"] == "negative")].sort_values("score", ascending=False).head(3)
print("\n--- TOP 3 NEGATIVE ARTICLES FROM NL ---")
for idx, row in nl_negative.iterrows():
    print(f"\nExample {idx} (score: {row['score']:.4f}):")
    print(row["text"][:2000])


--- TOP 3 NEGATIVE ARTICLES FROM NL ---

Example 94990 (score: 0.9665):
Mandag 19.august troppet 12 elever opp på Malangseidet skole, klar for et nytt skoleår. Både de og foreldrene var trygge på at de skulle få et år til på den nedleggingstruede skolen. Torsdag 22. august fikk vi melding om at skolen var blitt stengt med øyeblikkelig virkning. Tromsø brann og redning måtte stenge skolen pga. tekniske avvik som ikke var blitt rettet opp av Balsfjord kommune. Dette kom som lyn fra klar himmel på oss foreldre.
Hvordan i alle dager kunne dette skje uten at vi foreldre fikk noe informasjon? Administrasjonen fikk vite om dette november 2018. Jeg vil tro at de allerede i vår skjønte at de ikke ville klare å lukke avvikene før skolestart i høst. Burde de ikke da ha kalt inn til møte før sommerferien slik at elevene, foreldre, lærere og assistenter kunne forberedt seg på en skolestart på Sand skole. Elever har grått, foreldre har grått og lærere har grått. Det er så respektløs overfor dem og 

### *What author has the articles with the most negative/positive sentiment from each source?*

In [17]:
# find the most positive and most negative author from each source
most_positive_authors = (
	analysis_df[analysis_df["label"] == "positive"]
	.sort_values(["source", "score"], ascending=[True, False])
	.groupby("source", as_index=False)
	.head(1)
)

most_negative_authors = (
	analysis_df[analysis_df["label"] == "negative"]
	.sort_values(["source", "score"], ascending=[True, False])
	.groupby("source", as_index=False)
	.head(1)
)

print("\nAuthors with the most positive articles by source:")
print(most_positive_authors[["source", "author", "score"]])

print("\nAuthors with the most negative articles by source")
print(most_negative_authors[["source", "author", "score"]])


Authors with the most positive articles by source:
       source                               author     score
14194      AA                          Lars Sørnes  0.982586
40492      AP         Aftenpostens Musikkredaksjon  0.991355
43046      BT                          Ola Bernhus  0.985152
47705      DA                      Reidar Spigseth  0.994403
74333      DB                        Sofie Braseth  0.982366
66785      DN                    Jonas Christensen  0.950690
90578      FV                         Oakland Rain  0.990307
92636      NL                       Østen Mortvedt  0.952465
106841     SA                         Jan Askelund  0.993722
4776       VG  Camilla Norli, Gisle Oddstad (foto)  0.986848

Authors with the most negative articles by source
       source                                 author     score
14694      AA                                    NTB  0.813273
28368      AP                            Kenneth Moe  0.936094
44978      BT                 Eduardo

In [18]:
most_positive_article = analysis_df[analysis_df["score"] == analysis_df["score"].max()].iloc[0]
most_negative_article = analysis_df[analysis_df["score"] == analysis_df["score"].min()].iloc[0]

for article in [most_positive_article, most_negative_article]:
    print("\n" + "="*80)
    print(f"Source: {article['source']}")
    print(f"Author: {article['author']}")
    print(f"Sentiment Score: {article['score']:.4f}")
    print(article["text"][:2000])


Source: DA
Author: Reidar Spigseth
Sentiment Score: 0.9944
TV-DRAMA
HBO Nordic, premiere mandag
Den første sesongen av «True Detective» var et sjeldent mørkt og rystende krimdrama, preget av en stille, søvnig og knugende stemning. Serien var rett og slett skummel, og hovedrollene var besatt av skuespillere Woody Harrelson og Matthew McConaughey, som var nærmest naturskapt for det gudsforlatte, gjengrodde Sørstats-Amerika de befant seg i. Handlingen ble fortalt i to tidsperioder, blant annet med eldre, slitnere Rust Cohle i samtale om fortidens drapsetterforskning. Rollen ga McConaughey status som karakterskuespiller for en stakket stund.
Dette var stor fortellerkunst, noir på en ny måte, en TV-krim et nivå over det meste i sin sjanger og TV-drama generelt. Her produserte HBO TV av ypperste kvalitet igjen! Serieskaper Nic Pizzolatto fikk ry som en fornyer og genial serieskaper. Alt skrytet forsvant med sesong to i antologien, en mørk, men mer tradisjonelt fortalt politiserie.
Serien ha

---
### *Topics and Text Sentiment*

In [19]:
model_name = "NbAiLab/nb-sbert-base"
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)

model_dir = Path("../bertopic_models/NbAiLab_nb-sbert-base/NbAiLab_nb-sbert-base_ngram1-3_n50000_min_df10_20260409_201329").resolve()
topic_model = BERTopic.load(str(model_dir), embedding_model=embedding_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1614.15it/s]
BertModel LOAD REPORT from: NbAiLab/nb-sbert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
docs = analysis_df["text"].tolist()

topics, _ = topic_model.transform(docs)

Batches: 100%|██████████| 313/313 [15:48<00:00,  3.03s/it]
2026-04-29 17:26:35,757 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


In [21]:
# print all topics with their top 10 words except for topic -1 (outliers)
for topic in topic_model.get_topics().keys():
    if topic != -1:
        print(f"Topic {topic}: {', '.join([word for word, _ in topic_model.get_topic(topic)[:10]])}")

Topic 0: scoret, league, spillerne, spillere, champions, ballen, kamper, manchester, laget, sesong
Topic 1: brexit, parlamentet, statsministeren, statsminister, stortinget, regjering, valget, regjeringen, valg, avtalen
Topic 2: pågrepet, etterforskningen, drept, døde, fornærmede, politidistrikt, politiets, politiet, elisabeth, død
Topic 3: ulykken, nødetatene, politiet på, operasjonsleder, brannvesenet, politiet, politidistrikt, kjørte, brannen, hendelsen
Topic 4: årets, serien, tv, kjent, åringen, på, gang, filmen, fjor, film
Topic 5: president donald trump, presidentens, sa trump, presidenten, usas president, donald trump, president donald, trump, president, trumps
Topic 6: milliarder kroner, millioner kroner, kvartal, økte, veksten, milliarder, prisene, fjor, vekst, utviklingen
Topic 7: hunder, hund, dyrene, hunden, dyr, smittet, sykdommen, ulv, syke, diaré
Topic 8: meteorologisk, nedbør, meteorologisk institutt, stormen, været, regn, vind, varmere, vær, meldt
Topic 9: bompenger, el

In [22]:
analysis_df["topic"] = topics
analysis_df.head(3)

,file,url,source,date,author,gender,class1,class2,language,title,...,text,word_count,sentence_count,verb_ratio,subject_ratio,passive_ratio,passive_of_verbs,label,score,topic
5643,VG-20190423-ekstra-bladet-danske.xml,http://www.vg.no/nyheter/utenriks/i/awOaE7/eks...,VG,2019-04-23 12:57,"Ingvill Dybfest Dahl, Ingeborg Huse Amundsen, ...",Female,"nyheter,utenriks",None,Bokmål,Ekstra Bladet: Danske Anders Holch Povlsen på ...,...,"Det skriver danske Ekstra Bladet, som siterer ...",439,27,0.099804,0.088063,0.007828,0.078431,neutral,0.540266,2
6808,VG-20190516-sarah-jessica-parker.xml,http://www.vg.no/rampelys/film/i/rA5exK/sarah-...,VG,2019-05-16 07:37,Catherine Gonsholt Ighanian,Female,"rampelys,film",None,Bokmål,Sarah Jessica Parker raser mot ukeblad – viser...,...,Den tidligere «Sex og singelliv»-stjernen har ...,435,26,0.095785,0.086207,0.001916,0.020000,neutral,0.640248,4
4593,VG-20190401-rotteracet-om-champi.xml,http://www.vg.no/sport/fotball/i/opEPLm/rotter...,VG,2019-04-01 10:20,Ole Kristian Strøm,Male,"sport,fotball",None,Bokmål,Rotteracet om Champions League-plass: – Det er...,...,Mandagens oddstips finner du her\n– Jeg våger ...,611,45,0.098571,0.092857,0.004286,0.043478,neutral,0.488746,0


---
### *Combining topics, sentiment and linguistics*

In [23]:
display_topic = 4
display_label = "negative"

sentiment_topic_selected = (
    analysis_df[analysis_df["topic"] == display_topic]["label"].value_counts()
)

selected_topic_df = analysis_df[analysis_df["topic"] == display_topic]
female_author_count = selected_topic_df[selected_topic_df["gender"] == "Female"].shape[0]
male_author_count = selected_topic_df[selected_topic_df["gender"] == "Male"].shape[0]
unknown_author_count = selected_topic_df[selected_topic_df["gender"] == "Unknown"].shape[0]

print(f"Total author distribution in topic {display_topic}:")
print(f"Female authors: {female_author_count}")
print(f"Male authors: {male_author_count}")
print(f"Unknown authors: {unknown_author_count}")

# print how many female/male/unknown authors have the selected sentiment in the selected topic
gender_counts = selected_topic_df[
    (selected_topic_df["label"] == display_label)
]["gender"].value_counts(dropna=False)
female_authors = int(gender_counts.get("Female", 0))
male_authors = int(gender_counts.get("Male", 0))
unknown_authors = int(gender_counts.get("Unknown", 0))

print(f"\nGender distribution in topic {display_topic} with {display_label} sentiment:")
print(f"Female authors: {female_authors}")
print(f"Male authors: {male_authors}")
print(f"Unknown authors: {unknown_authors}")

print(f"\nSentiment distribution in topic {display_topic}:")
for label, count in sentiment_topic_selected.items():
    print(f"{label}: {count}")

print(f"\nTop 10 articles with {display_label} sentiment score in topic {display_topic}:")

for _, row in selected_topic_df[selected_topic_df["label"] == display_label].sort_values("score", ascending=False).head(5).iterrows():
    print(f"Title: {row['title']}, Source: {row['source']}, Author: {row['author']}, Score: {row['score']:.4f}")
    print(f"Text preview: {row['text'][:500]}")
    print("-" * 80)

Total author distribution in topic 4:
Female authors: 216
Male authors: 318
Unknown authors: 123

Gender distribution in topic 4 with negative sentiment:
Female authors: 4
Male authors: 11
Unknown authors: 2

Sentiment distribution in topic 4:
neutral: 381
positive: 259
negative: 17

Top 10 articles with negative sentiment score in topic 4:
Title: Hetset Northug og Klæbo – nå anklager han NRK for løftebrudd, Source: FV, Author: Pål Strande Gamlemoen,Anne-Marit Dahl, Score: 0.9363
Text preview: SEEFELD/OSLO: Halfvarsson har sammen med NRK og humorprogrammet Helt Ramm laget en rapvideo med tittelen «Who the fuck is Norway?»
Halfvarsson innleder videoen med å sage hodet av en pappfigur av Johannes Høsflot Klæbo. Han harselerer også med Petter Northugs fyllekjøring:
«Du kræsjet karrieren din som du kræsjet bilen din,» rapper den svenske skiløperen.
Martin Johnsrud Sundby og Norge som land får også høre det i videoen.
LES OGSÅ: Carragher innrømmer gigantisk feilvurdering: – En åpenbaring u


---
### *Linguistic features*

In [24]:
selected_topic = 4
selected_topic_df = analysis_df[analysis_df["topic"] == selected_topic]

for source in selected_topic_df["source"].unique():
        print(f"\nLinguistic features for topic {selected_topic} by {source}:")
        print(f"Average subject ratio: {selected_topic_df[selected_topic_df['source'] == source]['subject_ratio'].mean():.4f}")
        print(f"Average passive ratio: {selected_topic_df[selected_topic_df['source'] == source]['passive_ratio'].mean():.4f}")
        print(f"Average verb ratio: {selected_topic_df[selected_topic_df['source'] == source]['verb_ratio'].mean():.4f}")
        print(f"Average passive per verb: {selected_topic_df[selected_topic_df['source'] == source]['passive_of_verbs'].mean():.4f}")


Linguistic features for topic 4 by VG:
Average subject ratio: 0.0910
Average passive ratio: 0.0039
Average verb ratio: 0.0986
Average passive per verb: 0.0419

Linguistic features for topic 4 by AA:
Average subject ratio: 0.0889
Average passive ratio: 0.0034
Average verb ratio: 0.1006
Average passive per verb: 0.0353

Linguistic features for topic 4 by AP:
Average subject ratio: 0.0857
Average passive ratio: 0.0045
Average verb ratio: 0.0940
Average passive per verb: 0.0476

Linguistic features for topic 4 by BT:
Average subject ratio: 0.0824
Average passive ratio: 0.0033
Average verb ratio: 0.0911
Average passive per verb: 0.0375

Linguistic features for topic 4 by DA:
Average subject ratio: 0.0931
Average passive ratio: 0.0041
Average verb ratio: 0.0994
Average passive per verb: 0.0423

Linguistic features for topic 4 by DN:
Average subject ratio: 0.0844
Average passive ratio: 0.0020
Average verb ratio: 0.0967
Average passive per verb: 0.0217

Linguistic features for topic 4 by DB:


In [25]:
# print articles by AA in topic 16
aa_articles = analysis_df[(analysis_df["topic"] == 16) & (analysis_df["source"] == "NL")]
aa_articles = aa_articles[["title", "author", "source", "gender", "text", "verb_ratio", "subject_ratio", "passive_ratio", "passive_of_verbs", "topic"]]
aa_articles.head()

,title,author,source,gender,text,verb_ratio,subject_ratio,passive_ratio,passive_of_verbs,topic
93833,Naturvernforbundet – null troverdighet!,Tore Svendsen,NL,Male,Stortinget har vedtatt at Andøya Flystasjon sk...,0.111807,0.093023,0.005367,0.048000,16
95471,SAS tilbake med ny rute etter 12 års fravær,Are Medby,NL,Male,SAS kommer tilbake til Bardufoss - 12 år etter...,0.074074,0.080247,0.000000,0.000000,16
93340,Folke(helse)festen,Nordlys,NL,Unknown,Eventyret startet høsten 1989. Ideen om et mar...,0.102174,0.095652,0.004348,0.042553,16
92673,Nødraketter fra Gaza,Jarle Bakke,NL,Male,Mens Melodi Grand Prix forberedes med stor fes...,0.083333,0.085271,0.005814,0.069767,16
93904,Russisk militærfly skal overvåke i Nord-Norge,Pål Guttormsen,NL,Male,En russisk Antonov-maskin av typen An-30B skal...,0.133333,0.116667,0.008333,0.062500,16


In [26]:
# print full text from the first article by AA in topic 16
first_aa_article = aa_articles.iloc[3]
print(f"Text: {first_aa_article['text']}")
print(f"Verb Ratio: {first_aa_article['verb_ratio']:.4f}")
print(f"Subject Ratio: {first_aa_article['subject_ratio']:.4f}")
print(f"Passive Ratio: {first_aa_article['passive_ratio']:.4f}")
print(f"Passive per Verb: {first_aa_article['passive_of_verbs']:.4f}")

Text: Mens Melodi Grand Prix forberedes med stor festivitas i Israel, skytes primitive angreps-raketter opp fra Gazastripa. For de som kjenner palestinernes skjebne og lidelser under den israelske okkupasjonen, og virkningene av de store israelske militære bombeoffensiver fra 2007 og fram til i dag, kan oppskytingene moralsk betegnes som «nødraketter», med bønn til verden om hjelp.
Kontrasten mellom Midtøstens sterkeste militærmakt, Israel, med atomvåpen og USA som sin beste venn, og en utsultet og forpint befolkning som lever innesperret bak piggtråd, isolert fra omverden er et grovt brudd på elementære menneskerettigheter. At Israel ikke ser den historiske parallell til sin egen lidelseshistorie, vitner om hvor dypt avhumaniseringen er kommet og hvor umulig det er å lære av historien.
På Gazastripa, som er 36 km2, et område i utstrekning tilsvarende Målselvdalføret fra Øverbygd til Bardufoss, bor det 2 millioner mennesker. Halvparten av befolkningen er barn. Antallet drepte barn i fø